*2026 Spring DSAA 2011 Machine Learning*  
## Lab Note 13
*Changkai MAI*  
*Hong Kong University of Science and Technology (Guangzhou)*


## Question 1: Uncertainty Sampling

In this question, you will implement and compare **uncertainty-based query strategies** for Active Learning using a simple **multi-class** classification problem.

You are provided with:

* A 2D dataset with three classes, so the queried samples can be visualized clearly
* A fixed classifier, Logistic Regression
* An initial labeled set and a large unlabeled pool
* A complete active learning framework

Your goal is to understand how different uncertainty scores select different unlabeled samples and how quickly they improve test accuracy.

Your task is to complete the following steps.


In [ ]:
#====== Do not modify this block ======

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(2011)

# ===============================
# Dataset (2D for visualization)
# ===============================

X, y = make_blobs(
    n_samples=900,
    centers=[
        (-2, 0),
        (2, 0),
        (0, 2.5)
    ],
    cluster_std=[1.0, 1.0, 1.0],
    random_state=2011
)

X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2011
)

# initial labeled / unlabeled split
n_initial = 20
indices = np.random.permutation(len(X_train_pool))

labeled_idx = indices[:n_initial]
unlabeled_idx = indices[n_initial:]

X_L = X_train_pool[labeled_idx]
y_L = y_train_pool[labeled_idx]

X_U = X_train_pool[unlabeled_idx]
y_U = y_train_pool[unlabeled_idx]  # oracle (hidden to learner)

# ===============================
# Visualization utilities
# ===============================

def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.2)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=20, edgecolor="k")
    plt.title(title)

### Task 1: Implement Uncertainty Sampling Strategies

Implement the following three query strategies using the model's predicted probabilities:

1. **Least Confident Sampling:**  
   Select the sample whose maximum class probability is the lowest.

   $$
   x^* = \arg\max_x \left(1 - \max_y P_{\theta}(y | x) \right)
   $$

2. **Smallest Margin Sampling:**  
   Select the sample with the smallest difference between the two highest predicted class probabilities.

   $$
   x^* = \arg\min_x \left(P_{\theta}(y_1 | x) - P_{\theta}(y_2 | x) \right)
   $$

   Here $y_1$ is the most probable label of $x$, and $y_2$ is the second most probable label.

3. **Entropy Sampling:**  
   Select the sample with the highest entropy of predicted probabilities.

   $$
   x^* = \arg\max_x \left( -\sum_y P_{\theta}(y | x) \log P_{\theta}(y | x) \right)
   $$

Each strategy should return the **index of the selected sample** from the unlabeled pool.

**Hint:** Use only `model.predict_proba(X_pool)`. `np.argmax`, `np.argmin`, and `np.sort` are enough.


In [ ]:
# ===============================
# Query Strategies (Task 1)
# ===============================

def least_confident_sampling(model, X_pool):
    """
    Least Confident Sampling
    x* = argmax_x (1 - max_y P(y|x))
    """
    probs = model.predict_proba(X_pool)

    # TODO: compute uncertainty = 1 - maximum predicted class probability
    uncertainty = None

    # TODO: return the index of the most uncertain sample
    return None


def smallest_margin_sampling(model, X_pool):
    """
    Smallest Margin Sampling
    margin = P(y1|x) - P(y2|x)
    """
    probs = model.predict_proba(X_pool)

    # TODO: sort class probabilities in descending order for each sample
    sorted_probs = None

    # TODO: compute the margin between the largest and second-largest probabilities
    margin = None

    # TODO: return the index of the smallest margin
    return None


def entropy_sampling(model, X_pool):
    """
    Entropy Sampling
    entropy = - sum_y p(y|x) log p(y|x)
    """
    probs = model.predict_proba(X_pool)

    # TODO: compute entropy for each sample. Add a small number inside log.
    entropy = None

    # TODO: return the index of the largest entropy
    return None


def random_sampling(model, X_pool):
    return int(np.random.randint(len(X_pool)))


---

### Task 2: Integrate with the Active Learning Loop

Plug your query strategies into the provided active learning loop.

At each iteration:

* Train the model on the labeled data
* Select one sample from the unlabeled pool
* Query its label (oracle)
* Update the labeled set and the unlabeled pool

In [ ]:
# ===============================
# Active Learning Loop (Task 2)
# ===============================

def run_active_learning(query_fn, T=40, visualize_steps=None, title_prefix="Active Learning"):
    if visualize_steps is None:
        visualize_steps = []

    X_L_cur = X_L.copy()
    y_L_cur = y_L.copy()
    X_U_cur = X_U.copy()
    y_U_cur = y_U.copy()
    acc_history = []
    query_trace = []
    model = LogisticRegression(max_iter=1000, random_state=2011)

    for t in range(T):
        model.fit(X_L_cur, y_L_cur)
        acc = accuracy_score(y_test, model.predict(X_test))
        acc_history.append(acc)

        # TODO: query for a new sample index from X_U_cur
        idx = None

        # TODO: get the new sample and its oracle label
        x_new = None
        y_new = None
        query_trace.append((x_new[0].copy(), int(y_new[0]), float(acc)))

        if t in visualize_steps:
            plt.figure(figsize=(5.5, 4.2))
            plot_decision_boundary(
                model,
                np.vstack([X_L_cur, X_U_cur]),
                np.hstack([y_L_cur, y_U_cur]),
                title=f"{title_prefix}: iteration {t}, labeled={len(X_L_cur)}"
            )
            plt.scatter(x_new[:, 0], x_new[:, 1], c="red", marker="*", s=240, label="Queried")
            plt.legend()
            plt.tight_layout()
            plt.show()

        # TODO: move the queried sample from the unlabeled pool to the labeled set
        X_L_cur = None
        y_L_cur = None
        X_U_cur = None
        y_U_cur = None

    return np.array(acc_history), query_trace


### Task 3: Compare Learning Curves

Plot test accuracy versus the number of labeled samples for:

* Least Confident Sampling
* Smallest Margin Sampling
* Entropy Sampling
* Random Sampling (baseline)

All methods should be shown in a single figure for comparison.

In [ ]:
# ===============================
# Run Experiments
# ===============================

T = 80

# TODO: run active learning with different query strategies.
# Hint: use visualize_steps=[0, 10, 30] for one method only, so the notebook stays readable.
acc_LC, trace_LC = None, None
acc_SM, trace_SM = None, None
acc_ENT, trace_ENT = None, None
acc_rand, trace_rand = None, None

x_axis = np.arange(n_initial, n_initial + T)
plt.figure(figsize=(8, 5))
# TODO: plot all four learning curves in one figure.
plt.xlabel("# Labeled Samples")
plt.ylabel("Test Accuracy")
plt.title("Q1 Active Learning Performance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: print final accuracy and write a short analysis.


## Question 2: Query-By-Committee (QBC)

In this task, you will implement **Query-By-Committee (QBC)** as an alternative active learning strategy and compare it with uncertainty sampling.

Instead of using a single model, QBC selects samples based on **disagreement among multiple models**.

---

### Task 1: Build a Committee of Models

Construct a committee consisting of multiple classifiers.

* Use **Logistic Regression models** only.
* Each model should be trained on the same current labeled dataset.
* Introduce diversity using different regularization strengths and bootstrap samples.

Do not change the model family.

---

### Task 2: Implement a Disagreement Measure

Implement a query strategy based on **vote entropy**. Each model votes for a class label. Select the sample with the highest vote entropy.

---

### Task 3: Integrate QBC into the Active Learning Loop

At each iteration:

* Train all committee members on the labeled data, with bootstrap diversity
* Measure disagreement on the unlabeled pool
* Query one sample with maximum disagreement
* Update the labeled and unlabeled pools

---

### Task 4: Learning Curve Comparison

Run the experiment and plot test accuracy versus the number of labeled samples for:

* Query-By-Committee
* Entropy sampling with one model
* Random Sampling

All methods should appear in a single figure.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

data = load_iris()
X = data.data[:, :2]
q2_classes = np.unique(data.target)
y = data.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2011, stratify=y
)

n_initial = 20
rng_q2 = np.random.default_rng(2011)
indices = rng_q2.permutation(len(X_pool))
L_idx = indices[:n_initial]
U_idx = indices[n_initial:]
X_L, y_L = X_pool[L_idx], y_pool[L_idx]
X_U, y_U = X_pool[U_idx], y_pool[U_idx]

def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=18, edgecolor="k")
    plt.title(title)

def build_committee(n_models=7):
    # TODO: build LogisticRegression models with different regularization strengths C.
    committee = []
    return committee

def stratified_bootstrap_indices(y, rng):
    # TODO: sample indices with replacement inside each class.
    sampled = []
    return np.array(sampled, dtype=int)

def fit_committee(committee, X_train, y_train, rng):
    # TODO: fit each committee member on a bootstrap sample of the labeled set.
    pass

def committee_predict(committee, X):
    # TODO: predict with all models and return the majority vote for each sample.
    return None

def qbc_vote_entropy(committee, X_pool):
    # TODO: collect committee votes, convert vote counts to probabilities, compute entropy.
    return None

def random_qbc_sampling(committee, X_pool):
    return int(np.random.randint(len(X_pool)))

def single_model_entropy_sampling(model, X_pool):
    # TODO: implement entropy sampling for one LogisticRegression model.
    return None

def run_qbc(query_fn, T=30, visualize_steps=None):
    # TODO: train committee, query one point, update labeled/unlabeled pools, record accuracy.
    return None

def run_single_model_iris(query_fn, T=30):
    # TODO: implement a single-model active learning baseline.
    return None

T = 80
# TODO: compare QBC, single-model entropy, and random sampling in one plot.


## Question 3: Multi-Armed Bandits

In this task, you will study **Multi-Armed Bandits (MAB)** as a simplified form of reinforcement learning.

Unlike previous tasks, there is **no labeled dataset**. Instead, an agent repeatedly chooses one of several arms and receives a reward from the environment.

You are given a bandit environment with multiple arms.

* Each arm produces a stochastic reward.
* The true reward distributions are unknown to the agent.
* Interaction happens sequentially.

Your goal is to **design a strategy that balances exploration and exploitation**.

---

### Task 1: Implement Action-Selection Strategies

Implement the following bandit strategies:

1. **epsilon-greedy**
2. **Upper Confidence Bound (UCB)**

Each strategy should decide **which arm to pull** at each step.

---

### Task 2: Run Bandit Simulation

Simulate the bandit process for a fixed number of rounds.

At each round:

* Choose an arm
* Observe the reward
* Update your reward estimate
* Track expected regret using the true mean of the chosen arm

---

### Task 3: Compare Performance

Compare the strategies using:

* Average cumulative reward
* Average cumulative regret

Run multiple repetitions to reduce randomness and analyze the difference between strategies.

Focus on **strategy behavior and learning dynamics**, not parameter tuning.


In [ ]:
# ===========================================
# Bandit Environment (Given)
# ===========================================

class MultiArmedBandit:
    def __init__(self, means, rng=None):
        self.means = np.array(means, dtype=float)
        self.n_arms = len(means)
        self.rng = np.random.default_rng() if rng is None else rng

    def pull(self, arm):
        return self.rng.normal(self.means[arm], 1.0)

true_means = np.array([1.0, 1.3, 1.6, 1.1])
optimal_mean = float(np.max(true_means))

class EpsilonGreedyAgent:
    def __init__(self, n_arms, epsilon=0.1, rng=None):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.rng = np.random.default_rng() if rng is None else rng
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)

    def select_arm(self):
        # TODO: with probability epsilon choose a random arm; otherwise choose argmax estimated value.
        return None

    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        # TODO: incremental mean update: value <- value + (reward - value) / n
        pass

class UCBAgent:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        self.t = 0

    def select_arm(self):
        # TODO: select each arm once, then use value + sqrt(2 log(t+1) / count).
        return None

    def update(self, arm, reward):
        self.t += 1
        self.counts[arm] += 1
        n = self.counts[arm]
        # TODO: update estimated value of selected arm.
        pass

class RandomAgent:
    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.rng = np.random.default_rng() if rng is None else rng
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)

    def select_arm(self):
        return int(self.rng.integers(self.n_arms))

    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] += (reward - self.values[arm]) / n

def run_bandit(agent, bandit, T=500):
    rewards, regrets, actions = [], [], []
    # TODO: simulate T rounds.
    # Expected regret for chosen arm = optimal_mean - bandit.means[arm].
    return np.cumsum(rewards), np.cumsum(regrets), np.array(actions), agent

def average_runs(agent_factory, T=500, n_runs=50):
    # TODO: run multiple repetitions and average reward/regret curves.
    return None

T = 500
N_RUNS = 50
# TODO: compare epsilon-greedy, UCB, and random sampling.


## Question 4: Reinforcement Learning

### Background: CartPole Environment

#### Problem Overview

CartPole is a classical **control problem** from reinforcement learning.

The task is to **control a cart moving on a horizontal track** such that
a pole attached to the cart remains upright for as long as possible.

This is a **continuous-state, discrete-action** control problem.

---

#### Agent–Environment Interaction

At each time step, the interaction between the agent and the environment follows this loop:

1. The environment provides a **state** $s_t$
2. The agent selects an **action** $a_t$
3. The environment updates its dynamics
4. The agent receives a **reward** $r_t$
5. The environment transitions to a new state $s_{t+1}$

Your code does **not** control the physics directly.
It only decides which action to take.

---

#### State Space

In CartPole, the state is a 4-dimensional real-valued vector:

$$
s = (x,\ \dot{x},\ \theta,\ \dot{\theta})
$$

Where:

| Symbol           | Meaning                    |
| ---------------- | -------------------------- |
| $x$              | cart position              |
| $\dot{x}$        | cart velocity              |
| $\theta$         | pole angle (from vertical) |
| $\dot{\theta}$   | pole angular velocity      |

**Important**:

* You do not compute these values.
* They are provided by the environment at each step.

In code:

```python
state, _ = env.reset()
# state is a NumPy array of length 4
```
---

#### Action Space

The action space is **discrete**, with two possible actions:

| Action | Meaning                    |
| ------ | -------------------------- |
| 0      | push the cart to the left  |
| 1      | push the cart to the right |

At every time step, your agent must choose **exactly one** of these actions.

In code:

```python
action = 0 or 1
```

---

#### Environment Dynamics

The environment applies physical laws (Newtonian mechanics) to:

* update cart position and velocity
* update pole angle and angular velocity

The physical details are not covered in this lab.

Instead:

* treat the environment as a **black box**
* observe state transitions caused by your actions

---

#### Reward Function

CartPole uses a very simple reward definition:

$$
r_t =
\begin{cases}
1, & \text{if the pole is still upright} \
0, & \text{if the episode has terminated}
\end{cases}
$$

Interpretation:

> Each time step the pole does not fall, you receive +1 reward.

Therefore:

* Longer episodes → larger total reward
* Faster failure → smaller total reward

---

#### Episode Termination Conditions

An episode ends if **any** of the following conditions is met:

1. The pole angle $|\theta|$ becomes too large
2. The cart position $|x|$ moves too far from the center
3. The maximum episode length is reached (environment-dependent)

In code:

```python
next_state, reward, done, truncated, info = env.step(action)
```

* `done == True` indicates termination due to failure
* `truncated == True` indicates termination due to time limit

---

#### Objective of the Agent

The agent’s objective is to maximize the **cumulative reward**:

$$
\sum_{t=0}^{T} r_t
$$

Because the reward is +1 per step, this is equivalent to **keeping the pole balanced for as many steps as possible**.

---

#### Why CartPole Is a Useful Benchmark

CartPole is widely used in reinforcement learning because:

* it has low-dimensional state space
* it requires non-trivial control
* simple methods (Q-learning) partially work
* advanced methods (neural policies, PPO) work very well

This makes it ideal for algorithm comparison.

---

#### Summary (For Implementation)

From an implementation perspective:

* CartPole provides states $\{s\}$
* Your algorithm selects actions $\{a\}$
* The environment computes $s'$ and $r$
* Your code updates its internal parameters
* Performance is measured by episode length

Everything you implement fits into this loop.

In the lab, CartPole serves as a common testbed to compare different learning algorithms under the same conditions.

### Setup & Configuration

The following code provides the setup required to run the CartPole environment using Gymnasium. We also provide utility functions for visualization and evaluation.

The training budgets are intentionally small enough for a lab notebook. Results may vary slightly across machines because reinforcement learning is stochastic.


In [ ]:
#==============================
# Setup & Configuration
#==============================
import gymnasium as gym
import random
import warnings
import imageio
from IPython.display import Image, display

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning,
)

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    optim = None
    TORCH_AVAILABLE = False

SEED = 2011
EPISODES = 300
MAX_STEPS_PER_EPISODE = 500
PPO_TIMESTEPS = 20_000
MOVING_WINDOW = 20

FPS = 30
MAX_SECONDS = 8
ANGLE_THRESHOLD = 12 * np.pi / 180  # 12 degrees

np.random.seed(SEED)
random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    torch.set_num_threads(1)
    print(f"Using torch {torch.__version__}")
else:
    print("Torch unavailable in this runtime. Q-learning will run; neural policy and PPO will be unavailable.")


In [ ]:
#=============================
# Visualization Utilities
#=============================

def moving_average(x, window):
    x = np.asarray(x, dtype=float)
    if len(x) < window:
        return np.array([])
    return np.convolve(x, np.ones(window) / window, mode="valid")

def record_gif(policy_fn, filename):
    """Record a short CartPole rollout. GIF creation is optional for analysis."""
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    state, _ = env.reset(seed=SEED)
    env.action_space.seed(SEED)
    frames = []
    for step in range(MAX_SECONDS * FPS):
        frames.append(env.render())
        theta = state[2]
        if abs(theta) > ANGLE_THRESHOLD:
            break
        action = int(policy_fn(state))
        state, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            frames.append(env.render())
            break
    env.close()
    imageio.mimsave(filename, frames, fps=FPS)
    print(f"Saved GIF: {filename} ({len(frames)} frames)")
    display(Image(filename=filename))


### Task 1: Model-free learning (Q-learning)

#### Action-Value Function

Q-learning is based on the **action-value function**, defined as:

$$
Q(s, a) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \middle| s_0=s, a_0=a \right]
$$

Interpretation:

* $Q(s, a)$ estimates **how good it is to take action $a$ in state $s$**. 
* Assuming optimal actions are taken afterward.

---

#### Discrete Representation in Practice

In environments like CartPole:

* The state is continuous ($\mathbb{R}^4$)
* A table cannot store infinite states

Therefore, you **discretize the state space**:

$$
s \longrightarrow \hat{s} \in {0, \ldots, N_1-1} \times \cdots \times {0, \ldots, N_d-1}
$$

In code:

* You map each observation to a **tuple of bin indices**
* This tuple is used as an index into the Q-table

---

#### The Core Update Rule

The Q-learning algorithm updates Q-values using the **Bellman optimality update**:

$$
Q^*(s,a) \leftarrow Q(s,a) +
\alpha \Bigl(
r + \gamma \max_{a'} Q(s',a') - Q(s,a)
\Bigr)
$$

Where:

* $Q^*$ = updated Q-value
* $\alpha$ = learning rate
* $\gamma$ = discount factor
* $r$ = reward just received
* $s'$ = next state

This equation **is exactly what you implement in code**.

---

#### Meaning of Each Term (Directly Mapped to Code)

| Term               | Meaning              | In Code                 |
| ------------------ | -------------------- | ----------------------- |
| $Q(s,a)$         | current estimate     | `Q[state, action]`      |
| $r$             | immediate reward     | `reward`                |
| $\max_a Q(s',a)$ | best future estimate | `np.max(Q[next_state])` |
| $\alpha$         | step size            | `alpha`                 |
| $\gamma$         | future importance    | `gamma`                 |

The update moves the old estimate toward a **better estimate** based on experience.

---

#### Action Selection: ε-Greedy Policy

During training, actions are chosen using an **ε-greedy strategy**:

$$
a =
\begin{cases}
\text{random action}, & \text{with probability } \varepsilon \\
\arg\max_a Q(s,a), & \text{with probability } 1-\varepsilon
\end{cases}
$$

Purpose:

* prevent the agent from getting stuck in suboptimal actions
* ensure sufficient exploration

In code:

```python
if random.random() < epsilon:
    action = random_action()
else:
    action = argmax(Q[state])
```

---

#### Complete Algorithm

Your Q-learning code must implement the following loop:

* Initialize $Q(s,a) = 0$
* Repeat until step budget is exhausted:
   1. observe current state $s$
   2. discretize $s$
   3. select action using ε-greedy
   4. execute action, receive $r, s'$
   5. discretize $s'$
   6. update $Q(s,a)$ using the update rule above
* After training, the learned policy is:
   $$
   \pi(s) = \arg\max_a Q(s,a)
   $$

In [ ]:
#============================================
# Discretization Utility (Given)
#============================================

bins = [
    np.linspace(-4.8, 4.8, 9),
    np.linspace(-5, 5, 9),
    np.linspace(-0.418, 0.418, 9),
    np.linspace(-5, 5, 9),
]

def discretize(state):
    idx = [np.digitize(state[i], bins[i]) - 1 for i in range(4)]
    return tuple(int(np.clip(idx[i], 0, 8)) for i in range(4))

env = gym.make("CartPole-v1")
env.action_space.seed(SEED)
Q = np.zeros((9, 9, 9, 9, 2))
alpha, gamma_q = 0.1, 0.99
epsilon_start, epsilon_min, epsilon_decay = 1.0, 0.05, 0.985
returns_q = []

for episode in range(EPISODES):
    state, _ = env.reset(seed=SEED + episode)
    s = discretize(state)
    total = 0
    epsilon = max(epsilon_min, epsilon_start * (epsilon_decay ** episode))

    for _ in range(MAX_STEPS_PER_EPISODE):
        # TODO: choose an epsilon-greedy action from Q[s] using the decayed epsilon.
        action = None
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        s2 = discretize(next_state)
        # TODO: implement the Q-learning Bellman update.
        # target = reward if done else reward + gamma_q * max_a Q[s2, a]
        s = s2
        total += reward
        if done:
            break
    returns_q.append(total)

env.close()

def q_policy(state):
    return int(np.argmax(Q[discretize(state)]))


### Task 2: Learned Policy (MLP)

#### Motivation

In tabular Q-learning:

* the policy relies on a **Q-table**
* states must be **discrete**
* the table size grows rapidly with state dimension

In CartPole:

* the state is **continuous** and **4-dimensional**
* discretization loses precision
* learning becomes inefficient

Therefore, we replace the table with a **function approximator**.

---

#### Core Idea: Policy as a Function

Instead of storing values in a table, we represent the policy as a function:

$$
\pi_\theta(a \mid s)
$$

Where:

* $s$ is the state
* $a$ is an action
* $\theta$ are learnable parameters
* the output is a **probability distribution over actions**

An MLP is used to represent this function.

---

#### Input

The **input** to the policy network is the state vector:

$$
s = (x,\ \dot{x},\ \theta,\ \dot{\theta})
$$

In code:

```python
state.shape == (4,)
```

No discretization is applied.
The network operates directly on continuous values.

---

#### Output

The **output** of the MLP is a vector of action probabilities:

$$
\pi_\theta(\cdot \mid s) =
\begin{bmatrix}
P(a=0 \mid s) \\
P(a=1 \mid s)
\end{bmatrix}
$$

Properties:

* probabilities are non-negative
* probabilities sum to 1

This is typically implemented using a **Softmax** layer.

---

#### MLP Architecture

In this lab, the policy network has a simple structure:

$$
\text{State} \rightarrow \text{Linear} \rightarrow \text{ReLU}
\rightarrow \text{Linear} \rightarrow \text{Softmax}
$$

In code, this corresponds to:

* a small fully connected neural network
* usually one hidden layer (e.g. 32 units)

The architecture is **fixed** and **not a hyperparameter search problem**.

---

#### Action Selection

Given a state $s$:

1. Compute action probabilities using the network
2. Sample an action from the probability distribution

Formally:
$$
a_t \sim \pi_\theta(\cdot \mid s_t)
$$

In code:

* Use `dist = torch.distributions.Categorical(probs)` for probability distribution.
* Sampling method is implemented using `dist.sample()`.

---

#### Training

At timestep $T$, the **return** of a trajectory $\tau$ (a sequence of states, actions, rewards) is defined as:

$$
R(\tau) = \sum_{t=0}^{T} r_{t}
$$

Hence our objective is to maximize the expected return over all trajectories induced by the policy $\pi_\theta$:

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[R(\tau)\right] = \sum_{\tau} p_\theta(\tau) R(\tau)
$$

we have:

$$
p_{\theta}(\tau) = P(s_0) \prod_{t=0}^{T} \pi_\theta(a_t \mid s_t) P(s_{t+1} \mid s_t, a_t)
$$

where:
* $P(s_0)$ is the initial state distribution
* $P(s_{k+1} \mid s_k, a_k)$ is the environment dynamics

However, since the environment dynamics are unknown, we cannot compute this expectation directly. Instead, we use the **log-derivative trick**:

$$
\nabla_\theta p_{\theta}(\tau) = p_{\theta}(\tau) \nabla_\theta \log p_{\theta}(\tau)
$$

to get:

$$
\nabla_\theta J(\theta) = \sum_\tau p_{\theta}(\tau) \nabla_\theta \log p_{\theta}(\tau) R(\tau) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log p_{\theta}(\tau) R(\tau) \right]
$$

with:

$$
\log p_{\theta}(\tau) = \sum_{t=0}^{T} \log \pi_\theta(a_t \mid s_t) + \text{other terms not depending on } \theta
$$

This is the form of **policy gradient theorem**, which states that the gradient of the expected return with respect to the policy parameters $\theta$ is given by:

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta}} \left[\sum_{t = 0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) R(\tau) \right]
$$

Since we want to maximize $J(\theta)$, we define the ideal loss function $L(\theta) = -J(\theta)$, hence we have $\nabla_\theta L(\theta) = -\nabla_\theta J(\theta)$. Futhermore, We can exapnd $R(\tau)$:
$$
\nabla_\theta L(\theta) = - \mathbb{E}_{\tau \sim \pi_{\theta}} \left[\sum_{t = 0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \sum_{k=0}^{T} r_k \right] = - \sum_{t = 0}^T \sum_{k = 0}^T \mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) r_k \right]
$$

Notice that when $k < t$, $r_k$ becomes a fixed constant at timestep $t$, thus we have:
$$ 
\mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) r_k \right] = \mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) \right]\cdot r_k
$$ 

We already know that the choice of $a_t$ is independent of previous rewards $r_k$ for $k < t$, and only depends on the current state $s_t$, thus we have:
$$
\mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) \right] = \sum_{a_t} \pi_\theta(a_t \mid s_t) \nabla_\theta \log \pi_\theta(a_t \mid s_t) = \nabla_\theta \sum_{a_t} \pi_\theta(a_t \mid s_t) = \nabla_\theta 1 = 0
$$
Thus:
$$ 
\mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) r_k \right] = 0
$$ 
for $k < t$. Therefore, we can simplify the above equation as:
$$
\nabla_\theta L(\theta) = - \sum_{t = 0}^T \sum_{k = t}^T \mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) r_k \right]
$$

We define the **future return from time step $t$** as $G_t$, we have:
$$
\nabla_\theta L(\theta) = - \sum_{t = 0}^T \mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) G_t \right]   
$$

For an episode sampled from the policy $\pi_\theta$, we can approximate the expectation using a single sample, thus we have:
$$
\nabla_\theta L(\theta) \approx - \sum_{t = 0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t) G_t
$$

Finally, we can define our real loss function as:

$$
\mathcal{L}(\theta) = - \sum_{t=0}^{T} \log \pi_\theta(a_t \mid s_t) G_t
$$

This is the Monte Carlo approximation of the policy gradient loss function, which we will implement in code.

In [ ]:
if TORCH_AVAILABLE:
    env = gym.make("CartPole-v1")
    env.action_space.seed(SEED)
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    class PolicyNet(nn.Module):
        def __init__(self):
            super().__init__()
            # TODO: define a small MLP.
            # Input dimension: obs_dim
            # Output dimension: n_actions
            # Final layer should produce action probabilities.
            self.net = None

        def forward(self, x):
            # TODO: return action probabilities.
            return None

    policy = PolicyNet()
    optimizer = optim.Adam(policy.parameters(), lr=1e-2)
else:
    env = None
    policy = None
    optimizer = None

gamma_pg = 0.99


In [ ]:
returns_nn = []

if TORCH_AVAILABLE:
    for episode in range(EPISODES):
        # TODO:
        # 1. collect one episode using the policy network
        # 2. compute discounted future returns G_t
        # 3. normalize returns for stability
        # 4. update policy using REINFORCE loss: -sum log pi(a_t|s_t) * G_t
        pass
    env.close()
else:
    print("Neural policy unavailable because torch is not available in this runtime.")


def nn_policy(state):
    if not TORCH_AVAILABLE or policy is None:
        return q_policy(state)
    with torch.no_grad():
        probs = policy(torch.tensor(state, dtype=torch.float32))
    return int(torch.argmax(probs).item())


### PPO (For reference)

**Proximal Policy Optimization (PPO)** is a modern policy gradient method designed to improve the stability and efficiency of policy learning.

Unlike vanilla policy gradient methods, which can suffer from large, destabilizing updates, PPO constrains each policy update to stay close to the current policy. This is achieved by optimizing a *clipped surrogate objective* that penalizes overly large changes in action probabilities. 

As a result, PPO retains the conceptual simplicity of policy gradients while reducing the risk of destabilizing updates in control problems such as CartPole.

In practice, PPO is widely adopted because it often gives reliable results with modest tuning, provided that the interaction budget is large enough.

In [ ]:
if TORCH_AVAILABLE:
    try:
        from stable_baselines3 import PPO
        from stable_baselines3.common.monitor import Monitor

        env = Monitor(gym.make("CartPole-v1"))
        env.reset(seed=SEED)
        env.action_space.seed(SEED)
        ppo = PPO("MlpPolicy", env, verbose=0, seed=SEED, device="cpu")
        ppo.learn(total_timesteps=PPO_TIMESTEPS, progress_bar=False)
        returns_ppo = env.get_episode_rewards()
        env.close()
        PPO_AVAILABLE = True
        print(f"PPO: episodes={len(returns_ppo)}, mean_last10={np.mean(returns_ppo[-10:]):.1f}, best={np.max(returns_ppo):.1f}")

        def ppo_policy(state):
            return int(ppo.predict(state, deterministic=True)[0])

    except Exception as exc:
        PPO_AVAILABLE = False
        returns_ppo = []
        print(f"PPO skipped: {type(exc).__name__}: {exc}")

        def ppo_policy(state):
            return q_policy(state)
else:
    PPO_AVAILABLE = False
    returns_ppo = []
    print("PPO unavailable because torch or stable-baselines3 is not available in this runtime.")

    def ppo_policy(state):
        return q_policy(state)


### Visualization

In [ ]:
ma_q = moving_average(returns_q, MOVING_WINDOW)
ma_nn = moving_average(returns_nn, MOVING_WINDOW)
ma_ppo = moving_average(returns_ppo, MOVING_WINDOW)

plt.figure(figsize=(9, 4.5))
if len(ma_q) > 0:
    plt.plot(range(MOVING_WINDOW - 1, MOVING_WINDOW - 1 + len(ma_q)), ma_q, label="Q-learning", linewidth=2)
if len(ma_nn) > 0:
    plt.plot(range(MOVING_WINDOW - 1, MOVING_WINDOW - 1 + len(ma_nn)), ma_nn, label="Neural Policy", linewidth=2)
if len(ma_ppo) > 0:
    plt.plot(range(MOVING_WINDOW - 1, MOVING_WINDOW - 1 + len(ma_ppo)), ma_ppo, label="PPO", linewidth=2)
plt.xlabel("Episode")
plt.ylabel("Average Return")
plt.title("Q4 CartPole Learning Curves (Moving Average)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Q4 summary:")
print(f"  Q-learning    : final={returns_q[-1]:.1f}, mean_last20={np.mean(returns_q[-20:]):.1f}, best={np.max(returns_q):.1f}")
if len(returns_nn) > 0:
    print(f"  Neural policy : final={returns_nn[-1]:.1f}, mean_last20={np.mean(returns_nn[-20:]):.1f}, best={np.max(returns_nn):.1f}")
else:
    print("  Neural policy unavailable in this runtime.")
if len(returns_ppo) > 0:
    print(f"  PPO           : episodes={len(returns_ppo)}, mean_last10={np.mean(returns_ppo[-10:]):.1f}, best={np.max(returns_ppo):.1f}")
else:
    print("  PPO unavailable in this runtime.")

print("\nDetailed analysis:")
print("- Q-learning uses a discretized Q-table, so it loses information from the continuous CartPole state.")
print("- Epsilon decay helps Q-learning explore early and exploit more later, but tabular discretization is still limited.")
if len(returns_nn) > 0:
    print("- The neural policy uses the continuous state directly and can reach much longer episodes, but REINFORCE remains noisy under this small training budget.")
else:
    print("- The neural policy is skipped when torch is unavailable, but the code path is ready when torch is installed.")
if len(returns_ppo) > 0:
    print("- PPO has the strongest late-stage performance in this run after 20,000 timesteps, because clipped policy updates make training more stable.")
else:
    print("- PPO is skipped when stable-baselines3 is unavailable; conceptually, it improves policy-gradient stability with clipped updates.")


In [ ]:
gif_jobs = [
    ("q.gif", q_policy, True),
    ("nn.gif", nn_policy, TORCH_AVAILABLE and len(returns_nn) > 0),
    ("ppo.gif", ppo_policy, PPO_AVAILABLE),
]

for filename, policy_fn, enabled in gif_jobs:
    if not enabled:
        print(f"Skipped GIF: {filename} because the corresponding policy is unavailable.")
        continue
    try:
        record_gif(policy_fn, filename)
    except Exception as exc:
        print(f"Skipped GIF: {filename} ({type(exc).__name__}: {exc})")
